In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/andrewmvd/animal-faces")

In [ ]:
import torch
from sklearn.preprocessing import LabelEncoder # Label Encoder to encode the classes from strings to numbers
import torch  # Main PyTorch Library
import torchvision.transforms as transforms  # Transform function used to modify and preprocess all the images
from torch.utils.data import Dataset, DataLoader # Dataset class and DataLoader for creating the objects
from PIL import Image # Used to read the images from the directory
from torch import nn
from torchsummary import summary
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu" # detect the GPU if any, if not use CPU, change cuda to mps if you have a mac
print("Device available: ", device)

In [ ]:
image_path = [] # Empty array where we will fill the paths of the images
labels = []

for i in os.listdir('/content/animal-faces/afhq/'):
  for label in os.listdir(f'/content/animal-faces/afhq/{i}/'):
    for image in os.listdir(f'/content/animal-faces/afhq/{i}/{label}/'):
      labels.append(label)
      image_path.append(f'/content/animal-faces/afhq/{i}/{label}/{image}')

In [ ]:
zip(image_path, labels)
data_df = pd.DataFrame(zip(image_path, labels), columns=["image_paths", "labels"])
data_df.head()

In [ ]:
	image_paths	                                        labels
0	/content/animal-faces/afhq/train/wild/flickr_w...	wild
1	/content/animal-faces/afhq/train/wild/flickr_w...	wild
2	/content/animal-faces/afhq/train/wild/flickr_w...	wild
3	/content/animal-faces/afhq/train/wild/flickr_w...	wild
4	/content/animal-faces/afhq/train/wild/flickr_w...	wild


In [ ]:
train = data_df.sample(frac=0.7, random_state=1)
test = data_df.drop(train.index)

val = test.sample(frac=0.5, random_state=1)
test = test.drop(val.index)

In [ ]:
print(train)   
        
                                            image_paths labels
15582  /content/animal-faces/afhq/val/dog/pixabay_dog...    dog
5816   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
5657   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
7400   /content/animal-faces/afhq/train/dog/flickr_do...    dog
4097   /content/animal-faces/afhq/train/wild/flickr_w...   wild
...                                                  ...    ...
3622   /content/animal-faces/afhq/train/wild/flickr_w...   wild
14673  /content/animal-faces/afhq/val/wild/pixabay_wi...   wild
6021   /content/animal-faces/afhq/train/dog/pixabay_d...    dog
12669  /content/animal-faces/afhq/train/cat/pixabay_c...    cat
15396  /content/animal-faces/afhq/val/dog/pixabay_dog...    dog

[11291 rows x 2 columns]

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(data_df['labels'])

In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize((128, 128)), # Resize images to 128x128 as per the model summary input size
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)
])

Custom Dataset Class

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(label_encoder.transform(dataframe['labels'])).to(device)

    def __len__(self):
        return self.dataframe.shape[0]

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
          image = self.transform(image).to(device)

        return image, label

Create Dataset Objects


In [ ]:
train_dataset = CustomImageDataset(dataframe=train , transform=data_transforms)
val_dataset = CustomImageDataset(dataframe=val , transform=data_transforms)
test_dataset = CustomImageDataset(dataframe=test , transform=data_transforms)

DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16, shuffle=True)